# 15. 표본 11개 임베딩과 저장 전 확인

[프로젝트 적용] 기존 SentenceTransformer로 검색할 글을 숫자 벡터로 바꿉니다.
[프로젝트 추가] 토크나이저와 모델의 스냅샷 버전을 맞추고, 원문·그림·각주 연결과 글 식별값을 확인합니다.
기존 모델 jhgan/ko-sroberta-multitask-mrl을 로컬 CPU에서 사용합니다. 768차원·최대 128토큰이며 새 모델은 추가하지 않았습니다.
이 노트북에는 DB 저장이나 이미지 업로드 명령이 없습니다. 숫자 벡터는 메모리에만 남습니다.

In [ ]:
# [프로젝트 추가] 이동한 개인 코드 폴더를 기준으로 프로젝트와 src 경로를 찾습니다.
from pathlib import Path
import sys
current = Path.cwd().resolve()
project_folder = next((p for p in [current, *current.parents]
                       if (p / 'src/car_search_rag/zzong_santafe_lag/sample_embedding.py').is_file()), None)
if project_folder is None:
    raise FileNotFoundError('mle-02-p1-team1 프로젝트에서 실행하세요.')
src_path = str(project_folder / 'src')
if src_path not in sys.path:
    sys.path.insert(0, src_path)


In [ ]:
# PDF와 토크나이저를 읽고 표본의 DB 행을 메모리에 만듭니다. 아직 벡터는 없습니다.
from car_search_rag.zzong_santafe_lag.manual_service import ManualService
from car_search_rag.zzong_santafe_lag.db_rows import build_sample_rows
service = ManualService().prepare()
preview = build_sample_rows(service)
print('표본 건수:', preview.summary()['table_counts'])
print('모델 스냅샷:', service.token_counter.model_revision)


## 선택한 11개 청크를 숫자로 바꾸기

다음 셀은 실제 임베딩 계산을 합니다. 전체 2,213개를 계산하지 않습니다.
그림은 사람이 확인한 설명 글과 연결 경로를 사용하며 그림 픽셀 자체를 임베딩하는 것은 아닙니다.
같은 preview에 이미 벡터가 있으면 중복 계산을 막습니다. 재계산하려면 앞 셀에서 새 preview를 준비하세요.

In [ ]:
# [프로젝트 추가] 같은 버전의 로컬 모델을 읽고, 벡터를 원래 청크 순서로 연결합니다.
from car_search_rag.zzong_santafe_lag.sample_embedding import embed_sample_rows
preview = embed_sample_rows(preview, service)
report = preview.embedding_report
print('모델:', report['model_name'])
print('버전:', report['model_revision'])
print('벡터 크기:', report['shape'])
print('숫자가 유효한가요?:', report['finite'])
print('벡터 길이 범위:', report['norm_min'], '~', report['norm_max'])
print('DB에 저장했나요?:', report['db_written'])


In [ ]:
# 저장할 검색 글과 출처를 읽습니다. raw_text는 원문 그대로 별도 보관합니다.
for row in preview.tables['parent_records']:
    print('\n주제:', row['title'], '| PDF:', row['source_pages'])
    print('함께 읽을 기록:', row['metadata'].get('required_context_record_ids', []))
    print(row['content'])
# 벡터 전체를 출력하지 않고 첫 청크와 앞 숫자 5개만 확인합니다.
first = preview.tables['chunks'][0]
print('\n첫 청크:', first['content'])
print('글 식별값:', first['content_sha256'])
print('벡터 앞 5개:', first['embedding'][:5])


## 계산 완료와 DB 저장 완료의 차이

embedding_complete=True는 표본 벡터가 메모리에 연결됐다는 뜻입니다. ready_for_insert=False는 저장 전 검토·승인이 남아 있다는 뜻입니다. DB의 처리 작업 ready 상태도 아직 만들지 않았습니다.
실제 저장은 사용자 확인 후 별도 트랜잭션으로 진행합니다. 전체 기록 저장, 이미지 업로드와 DB 검색 확인은 이후 단계입니다.
커널을 종료하거나 다시 시작하면 이 벡터는 사라집니다. 이 노트북과 .py 실행 파일에는 벡터 파일 저장 기능이 없습니다.